In [1]:
# ================= CIFAR-10 | ADQ QAT (STABLE NO-DROP) =================
!pip install torch torchvision timm tqdm --quiet

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm
import torch.ao.quantization as quant
from timm.utils import ModelEma

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 100
WARMUP_EPOCHS = 40
BATCH_SIZE = 128

# ---------------- DATA ----------------
MEAN = [0.4914, 0.4822, 0.4465]
STD  = [0.2470, 0.2435, 0.2616]

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_ld = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_ld  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# ---------------- MODEL ----------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model("resnet18", pretrained=True, num_classes=10)

        base.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        base.maxpool = nn.Identity()

        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()

        self.stem = nn.Sequential(base.conv1, base.bn1, base.act1)

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4

        self.pool = base.global_pool
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        x = self.stem(x)

        x = self.quant(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.dequant(x)

        x = self.layer3(x)
        x = self.layer4(x)

        x = self.pool(x)
        return self.fc(x)

model = Net().to(device)
ema = ModelEma(model, decay=0.995)  # 🔥 stronger stability

# ---------------- OPTIM ----------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = optim.SGD(
    model.parameters(),
    lr=0.03,   # 🔥 reduced LR
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ---------------- EVAL ----------------
@torch.no_grad()
def evaluate(m):
    m.eval()
    correct = total = 0
    for x, y in test_ld:
        x, y = x.to(device), y.to(device)
        out = m(x)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

# ---------------- TRAIN ----------------
best_acc = 0
qat_enabled = False
best_weights = None

for epoch in range(EPOCHS):
    model.train()

    # 🔥 LR warmup
    if epoch < 5:
        for g in optimizer.param_groups:
            g["lr"] = 0.01

    loop = tqdm(train_ld, desc=f"Epoch {epoch+1}")
    for x, y in loop:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = criterion(out, y)

        # 🔥 pseudo ADQ
        if qat_enabled:
            commitment_loss = 0
            for m in model.modules():
                if isinstance(m, (nn.Conv2d, nn.Linear)):
                    w = m.weight
                    scale = w.abs().max().detach() + 1e-8
                    w_q = torch.clamp(w / scale, -1, 1)
                    commitment_loss += ((w - w_q.detach()) ** 2).mean()

            loss += 0.0005 * commitment_loss

        # 🔥 anti-forgetting anchor
        if best_weights is not None and best_acc > 0.75:
            anchor_loss = 0
            for name, p in model.named_parameters():
                anchor_loss += ((p - best_weights[name])**2).mean()
            loss += 1e-4 * anchor_loss

        optimizer.zero_grad()
        loss.backward()

        # 🔥 gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        ema.update(model)

    # 🔥 controlled scheduler
    if epoch >= 5:
        scheduler.step()

    acc = evaluate(ema.ema)
    print(f"Acc: {acc:.4f}")

    # 🔥 LR clamp after good performance
    if acc > 0.80:
        for g in optimizer.param_groups:
            g["lr"] = min(g["lr"], 0.005)

    # -------- ENABLE QAT --------
    if epoch + 1 == WARMUP_EPOCHS and not qat_enabled:
        print(">>> ENABLE QAT")

        model.qconfig = quant.get_default_qat_qconfig("fbgemm")
        quant.prepare_qat(model, inplace=True)

        # 🔥 freeze BN
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

        for g in optimizer.param_groups:
            g["lr"] = 1e-3

        qat_enabled = True

    # 🔥 save best
    if acc > best_acc:
        best_acc = acc
        best_weights = {k: v.detach().clone() for k, v in model.state_dict().items()}
        torch.save(ema.ema.state_dict(), "best_cifar10_adq_stable.pth")

# ---------------- CONVERT ----------------
model.cpu().eval()
quant.convert(model, inplace=True)

print("FINAL ACC:", best_acc)

100%|██████████| 170M/170M [00:01<00:00, 93.5MB/s]


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 391/391 [00:41<00:00,  9.45it/s]


Acc: 0.2058


Epoch 2: 100%|██████████| 391/391 [00:43<00:00,  9.07it/s]


...

Acc: 0.9371


Epoch 100: 100%|██████████| 391/391 [00:56<00:00,  6.98it/s]


Acc: 0.9368


/tmp/ipykernel_22/494408420.py:193: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quant.convert(model, inplace=True)


FINAL ACC: 0.938
